# Zephyr Medical Text Simplification 


In [1]:
# Cell 1 — Fix pandas FIRST (prevents circular import that breaks transformers)
# Run this cell, then Cell 2, then RESTART KERNEL before continuing
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pandas'], check=True)
print(' pandas reinstalled')

 pandas reinstalled


In [2]:
# Cell 2 — Install compatible package versions
# Pins transformers to 4.40.2 for compatibility with trl==0.8.6
!pip install -q \
    "transformers==4.40.2" \
    "trl==0.8.6" \
    "peft==0.11.0" \
    "accelerate==0.30.0" \
    "bitsandbytes>=0.43.0" \
    "datasets==2.19.0" \
    "sentencepiece==0.2.0"

import importlib
for pkg in ['transformers', 'trl', 'peft', 'accelerate', 'bitsandbytes', 'datasets']:
    try:
        v = importlib.import_module(pkg).__version__
        print(f'  {pkg}: {v}')
    except:
        print(f'   {pkg}: NOT found')

print('')
print('  NOW RESTART THE KERNEL (Run → Restart Kernel), then run from Cell 3 onward')

  transformers: 4.40.2
  trl: 0.8.6
  peft: 0.11.0
  accelerate: 0.30.0
  bitsandbytes: 0.49.2
  datasets: 2.19.0

  NOW RESTART THE KERNEL (Run → Restart Kernel), then run from Cell 3 onward


In [3]:
# Cell 3 — Imports & GPU check (run AFTER kernel restart)
import os, torch, json, shutil
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    GenerationConfig,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
    AutoPeftModelForCausalLM,
)
from trl import SFTTrainer
from datasets import load_dataset

# Version-safe SFTConfig fallback
import trl
print(f"trl version: {trl.__version__}")
try:
    from trl import SFTConfig
    USE_SFT_CONFIG = True
    print(" SFTConfig available")
except ImportError:
    SFTConfig = TrainingArguments
    USE_SFT_CONFIG = False
    print("SFTConfig not in this trl version — using TrainingArguments fallback")

print(' All imports OK')
if torch.cuda.is_available():
    print(f'   GPU  : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f'   bf16 supported: {bf16_ok} → training will use fp16={not bf16_ok}')
else:
    print(' No GPU — Settings → Accelerator → GPU P100')

2026-03-07 09:19:07.504001: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772875147.526776     223 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772875147.533518     223 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772875147.550981     223 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772875147.551006     223 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772875147.551009     223 computation_placer.cc:177] computation placer alr

trl version: 0.8.6
SFTConfig not in this trl version — using TrainingArguments fallback
 All imports OK
   GPU  : Tesla P100-PCIE-16GB
   VRAM : 17.1 GB
   bf16 supported: True → training will use fp16=False


In [4]:
# Cell 4 — Config
BASE_MODEL  = 'HuggingFaceH4/zephyr-7b-beta'
DATASET     = 'cbasu/Med-EASi'
OUTPUT_DIR  = '/kaggle/working/med-zephyr-final'
MAX_SEQ_LEN = 512
NUM_EPOCHS  = 3
BATCH_SIZE  = 4
GRAD_ACCUM  = 2       # effective batch = 8
LR          = 5e-5


USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = not USE_BF16

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config OK')
print(f'   Model  : {BASE_MODEL}')
print(f'   Output : {OUTPUT_DIR}')
print(f'   Epochs : {NUM_EPOCHS} | Batch : {BATCH_SIZE} | LR : {LR}')
print(f'   bf16   : {USE_BF16} | fp16 : {USE_FP16}')

Config OK
   Model  : HuggingFaceH4/zephyr-7b-beta
   Output : /kaggle/working/med-zephyr-final
   Epochs : 3 | Batch : 4 | LR : 5e-05
   bf16   : True | fp16 : False


In [5]:
# Cell 5 — Load & format dataset
dataset = load_dataset(DATASET)

def format_prompt(sample):
    sample['text'] = (
        '<|system|>\n'
        'You are a medical text simplification assistant. '
        'Rewrite complex medical text in simple language that patients can easily understand. '
        'Keep all key information but remove jargon.\n'
        '<|user|>\n'
        'Simplify this medical text:\n'
        f"{sample['Expert']}\n"
        '<|assistant|>\n'
        f"{sample['Simple']}"
    )
    return sample

dataset = dataset.map(format_prompt)
print(f' Dataset loaded')
print(f'   Train : {len(dataset["train"]):,}')
print(f'   Val   : {len(dataset["validation"]):,}')
print('\nExample preview:')
print(dataset['train']['text'][0][:300])

Using the latest cached version of the dataset since cbasu/Med-EASi couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /root/.cache/huggingface/datasets/cbasu___med-ea_si/default/0.0.0/10f8cb0f25f8aa4d74723f8b625e894ccaae8d3d (last modified on Sat Mar  7 06:34:31 2026).


 Dataset loaded
   Train : 1,397
   Val   : 196

Example preview:
<|system|>
You are a medical text simplification assistant. Rewrite complex medical text in simple language that patients can easily understand. Keep all key information but remove jargon.
<|user|>
Simplify this medical text:
75-90 % of the affected people have mild intellectual disability.
<|assist


In [6]:
# Cell 6 — Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.padding_side  = 'left'
tokenizer.pad_token     = tokenizer.eos_token
tokenizer.add_eos_token = True
print(' Tokenizer loaded')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /HuggingFaceH4/zephyr-7b-beta/resolve/main/tokenizer_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7b0eca3121b0>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: 142a203a-922a-43e9-bc8f-17fab26e97d0)')' thrown while requesting HEAD https://huggingface.co/HuggingFaceH4/zephyr-7b-beta/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /HuggingFaceH4/zephyr-7b-beta/resolve/main/tokenizer_conf

 Tokenizer loaded


In [7]:
# Cell 7 — Load model in 4-bit BitsAndBytes (NO GPTQ, NO optimum)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,   # float16 always safe on P100
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache      = False
model.config.pretraining_tp = 1
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

print(' Model loaded')
print(f'   VRAM used : {torch.cuda.memory_allocated(0)/1e9:.1f} GB')

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /HuggingFaceH4/zephyr-7b-beta/resolve/main/config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7b0eca3183e0>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: 1a44db6d-c9f9-4a66-959d-5289298bd192)')' thrown while requesting HEAD https://huggingface.co/HuggingFaceH4/zephyr-7b-beta/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /HuggingFaceH4/zephyr-7b-beta/resolve/main/config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7b0eca31a060>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: 3a6003e2-f09d-4247-9008-ef6557807a54)')' thrown while requesting HEAD https://huggingface.co/HuggingFac

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /HuggingFaceH4/zephyr-7b-beta/resolve/main/generation_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7b0ec91fe8d0>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: 73bf23c3-a796-419b-85eb-dffb5f8e6f2a)')' thrown while requesting HEAD https://huggingface.co/HuggingFaceH4/zephyr-7b-beta/resolve/main/generation_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /HuggingFaceH4/zephyr-7b-beta/resolve/main/generation_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7b0ec8f4d6a0>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: 13f41c5d-814b-41eb-b8d6-c02f9088d945)')' thrown while requesting HEAD 

 Model loaded
   VRAM used : 5.2 GB


In [8]:
# Cell 8 — LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print(' LoRA applied')

trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879
 LoRA applied


In [9]:
# Cell 9 — Training
training_kwargs = dict(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    optim='paged_adamw_32bit',
    learning_rate=LR,
    lr_scheduler_type='cosine',
    num_train_epochs=NUM_EPOCHS,
    bf16=USE_BF16,
    fp16=USE_FP16,
    evaluation_strategy='epoch',   # ← older name, works in all versions
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    push_to_hub=False,
    save_total_limit=2,
    report_to='none',
    warmup_ratio=0.1,
    weight_decay=0.01,
gradient_checkpointing_kwargs={"use_reentrant": False},
)

if USE_SFT_CONFIG:
    training_kwargs['dataset_text_field'] = 'text'
    training_kwargs['max_seq_length'] = MAX_SEQ_LEN
    training_kwargs['packing'] = False

sft_config = SFTConfig(**training_kwargs)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    tokenizer=tokenizer,           # ← works across all trl versions
    args=sft_config,
    **({} if USE_SFT_CONFIG else {
        'dataset_text_field': 'text',
        'max_seq_length': MAX_SEQ_LEN,
        'packing': False,
    })
)

print(' Starting training...')
trainer.train()
print(' Training complete!')

Map:   0%|          | 0/1397 [00:00<?, ? examples/s]

Map:   0%|          | 0/196 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(


 Starting training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,0.770000,0.858411
2,0.732200,0.852910
3,0.678900,0.858500


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /HuggingFaceH4/zephyr-7b-beta/resolve/main/config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7b0ea6e36ed0>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: 23a1bc86-d17a-400d-9e45-a5d6208ee116)')' thrown while requesting HEAD https://huggingface.co/HuggingFaceH4/zephyr-7b-beta/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /HuggingFaceH4/zephyr-7b-beta/resolve/main/config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7b0ea6e35cd0>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: 0fe48658-1fb0-494d-8b36-d9d10aeb1f4c)')' thrown while requesting HEAD https://huggingface.co/HuggingFac

 Training complete!


In [10]:
# Cell 10 — Save model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
trainer.save_state()

config_info = {
    'base_model'   : BASE_MODEL,
    'dataset'      : DATASET,
    'epochs'       : NUM_EPOCHS,
    'batch_size'   : BATCH_SIZE,
    'lr'           : LR,
    'lora_r'       : 16,
    'lora_alpha'   : 32,
    'quantization' : 'BitsAndBytes-nf4',
    'precision'    : 'bf16' if USE_BF16 else 'fp16',
}
with open(f'{OUTPUT_DIR}/training_config.json', 'w') as f:
    json.dump(config_info, f, indent=2)

shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print(f'Saved & zipped → {OUTPUT_DIR}.zip')
print('\nFiles:')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = f'{OUTPUT_DIR}/{fname}'
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1024**2
        print(f'  {fname:<45} {size:.1f} MB')

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /HuggingFaceH4/zephyr-7b-beta/resolve/main/config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7b0ec90d66c0>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: 3f63e538-b13b-4001-871b-c830cf88f1b5)')' thrown while requesting HEAD https://huggingface.co/HuggingFaceH4/zephyr-7b-beta/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /HuggingFaceH4/zephyr-7b-beta/resolve/main/config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7b0ec8c4af30>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: 209fb245-b108-40b7-97fb-d245f4157c52)')' thrown while requesting HEAD https://huggingface.co/HuggingFac

Saved & zipped → /kaggle/working/med-zephyr-final.zip

Files:
  README.md                                     0.0 MB
  adapter_config.json                           0.0 MB
  adapter_model.safetensors                     52.0 MB
  special_tokens_map.json                       0.0 MB
  tokenizer.json                                1.7 MB
  tokenizer.model                               0.5 MB
  tokenizer_config.json                         0.0 MB
  trainer_state.json                            0.0 MB
  training_args.bin                             0.0 MB
  training_config.json                          0.0 MB


In [12]:
# Cell 11 — Fix inference issues

model.eval()

# FIX 1: padding_side must be left for decoder-only models
tokenizer.padding_side = 'left'

# FIX 2: Strict prompt — no examples in context, stop at first response only
def simplify(medical_text):
    prompt = (
        '<|system|>\n'
        'You are a medical text simplification assistant. '
        'Rewrite the given medical text in simple language a patient can understand. '
        'Give only the simplified version. Do not repeat the instruction. Do not add extra information.\n'
        '<|user|>\n'
        f'Simplify this medical text:\n{medical_text}\n'
        '<|assistant|>\n'
    )
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,        # FIX 3: limit output length
            do_sample=False,           # greedy — more focused output
            temperature=1.0,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the NEW tokens (not the prompt)
    input_len = inputs['input_ids'].shape[1]
    new_tokens = outputs[0][input_len:]
    result = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # FIX 3: cut off if model starts repeating user/system tags
    for stop in ['<|user|>', '<|system|>', '<|assistant|>']:
        if stop in result:
            result = result.split(stop)[0]

    return result.strip()

tests = [
    'Most strabismus is caused by Refractive error; Muscle imbalance.',
    '75-90% of the affected people have mild intellectual disability.',
    'The patient presents with dyspnea, tachycardia, and bilateral pulmonary infiltrates.',
]

print('Inference Tests')
print('=' * 60)
for text in tests:
    print(f'ORIGINAL  : {text}')
    print(f'SIMPLIFIED: {simplify(text)}')
    print('-' * 60)

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Inference Tests
ORIGINAL  : Most strabismus is caused by Refractive error; Muscle imbalance.


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


SIMPLIFIED: 
------------------------------------------------------------
ORIGINAL  : 75-90% of the affected people have mild intellectual disability.


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


SIMPLIFIED: 
------------------------------------------------------------
ORIGINAL  : The patient presents with dyspnea, tachycardia, and bilateral pulmonary infiltrates.
SIMPLIFIED: 
------------------------------------------------------------
